# Evaluacion del Agente — Duckietown (Deep RL) — Google Colab

Evalua `best_duckie_agent.zip` (PPO) desde el repo y genera un video de conduccion.
Usa **Python 3.11** via deadsnakes + `xvfb-run` + `pyvirtualdisplay`, igual que `Fase2_DQN_PPO_v3.ipynb`.

> El profesor cambiara unicamente el mapa a `Duckietown-loop_obstacles-v0`.

## 1. Setup: clonar repo e instalar dependencias
Ejecutar una sola vez. Tarda ~5-10 min.

In [ ]:
import sys, os

REPO     = 'Aprendizaje-por-Refuerzo-y-Conducci-n-Aut-noma'
REPO_ABS = f'/content/{REPO}'
PY       = '/usr/bin/python3.11'

if not os.path.exists(REPO_ABS):
    os.system(f'git clone https://github.com/JaviCeronn/{REPO}.git {REPO_ABS}')
os.chdir(REPO_ABS)
os.system('git pull origin main')

print('[1/5] software-properties-common...')
os.system('sudo apt-get install -y -qq software-properties-common > /dev/null')
print('[2/5] PPA deadsnakes...')
os.system('sudo add-apt-repository -y ppa:deadsnakes/ppa > /dev/null 2>&1')
os.system('sudo apt-get update -qq')
print('[3/5] Python 3.11...')
os.system('sudo apt-get install -y -qq python3.11 python3.11-venv python3.11-dev python3.11-distutils > /dev/null')
os.system('wget -q https://bootstrap.pypa.io/get-pip.py -O /tmp/get-pip.py')
os.system('python3.11 /tmp/get-pip.py -q')
print('[4/5] OpenGL / Xvfb / glxinfo...')
# mesa-utils: incluye glxinfo, requerido por gym_duckietown/check_hw.py en Simulator.__init__
os.system('sudo apt-get install -y -qq xvfb freeglut3-dev libosmesa6-dev libgl1-mesa-dri libgl1-mesa-glx libglu1-mesa libturbojpeg mesa-utils > /dev/null')
print('[5/5] Paquetes Python en python3.11...')
# torch==2.12.0 no existe en PyPI; se filtra para que pip no aborte.
os.system("grep -v '^torch' requirements.txt > /tmp/req_notorch.txt")
os.system(f'{PY} -m pip install -q -r /tmp/req_notorch.txt')
os.system(f'{PY} -m pip install -q --no-deps git+https://github.com/duckietown/gym-duckietown.git@daffy')
# gymnasium>=0.26 puede subir pyglet a 2.x, incompatible con gym-duckietown daffy (API pyglet 1.x)
os.system(f'{PY} -m pip install -q "pyglet==1.5.27"')
import subprocess as _sp
_r = _sp.call([PY, '-c', 'import gym_duckietown, numpy; print(numpy.__version__)'])
print('Dependencias listas.' if _r == 0 else 'ERROR - revisa output arriba.')

## 2. Evaluacion y video

Todo el codigo de evaluacion esta embebido aqui (sin ficheros externos).
Cambia `MAP_NAME` si es necesario.

In [ ]:
import pathlib, subprocess, os
from IPython.display import Video, display

PY        = '/usr/bin/python3.11'
REPO_ABS  = '/content/Aprendizaje-por-Refuerzo-y-Conducci-n-Aut-noma'
MAP_NAME  = 'Duckietown-small_loop-v0'   # Evaluacion final: 'Duckietown-loop_obstacles-v0'
VIDEO_PATH = REPO_ABS + '/models/duckie_eval_video.mp4'

# RENDER HEADLESS POR SOFTWARE (no depende de la GPU; el subprocess no puede usar
# el OpenGL por hardware de la T4 y por eso segfaultea con el recipe de hardware).
# Esta es la unica config que creo el contexto GL sin SIGSEGV (llego hasta dentro
# de Simulator.__init__). Claves:
#   - xvfb-run con "+extension GLX": display X con protocolo OpenGL.
#   - LIBGL_ALWAYS_SOFTWARE=1: Mesa llvmpipe (render por CPU, siempre disponible).
#   - shadow_window=False: evita la ventana oculta GL de pyglet que peta en Xvfb.
#   - mesa-utils aporta glxinfo + parche check_hw por si acaso.
#   - Un solo display (sin pyvirtualdisplay, que arrancaria un 2o Xvfb sin GLX).
script = r"""
import os
os.environ['LIBGL_ALWAYS_SOFTWARE'] = '1'
os.environ['MESA_GL_VERSION_OVERRIDE'] = '3.3'
os.environ['MESA_GLSL_VERSION_OVERRIDE'] = '330'
os.environ['PYOPENGL_PLATFORM'] = 'glx'

import warnings, logging
warnings.filterwarnings('ignore')
logging.disable(logging.CRITICAL)

import pyglet
pyglet.options['shadow_window'] = False
pyglet.options['debug_gl'] = False

import numpy as np
import gym as old_gym
import gymnasium as gym
from gymnasium import spaces
import gym_duckietown

# Parche check_hw: bug en gym-duckietown -> si glxinfo no responde, b=None y b.decode() peta
import gym_duckietown.simulator as _simmod
def _safe_gfx():
    try:
        import subprocess as _sp
        b = _sp.check_output(['glxinfo', '-B'], stderr=_sp.STDOUT, timeout=5)
        return b.decode() if b else ''
    except Exception:
        return ''
_simmod.get_graphics_information = _safe_gfx

import cv2
import torch
import torch.nn as nn
import imageio
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor

IMG_SIZE  = 64
N_STACK   = 4
OBS_SHAPE = (1, IMG_SIZE, IMG_SIZE)

MAP_NAME   = 'Duckietown-small_loop-v0'
MODEL_PATH = '/content/Aprendizaje-por-Refuerzo-y-Conducci-n-Aut-noma/models/best_duckie_agent'
VIDEO_PATH = '/content/Aprendizaje-por-Refuerzo-y-Conducci-n-Aut-noma/models/duckie_eval_video.mp4'


class DuckieWrapper(gym.Env):
    metadata = {'render_modes': ['rgb_array']}

    def __init__(self, env_name=MAP_NAME, seed=None):
        super().__init__()
        self.env_name = env_name
        self.env = old_gym.make(env_name)
        if seed is not None:
            try:
                self.env.seed(seed)
            except Exception:
                pass
        self.action_space = spaces.Box(
            low=np.array([-1.0, -1.0], dtype=np.float32),
            high=np.array([1.0, 1.0], dtype=np.float32), dtype=np.float32)
        self.observation_space = spaces.Box(low=0, high=255, shape=OBS_SHAPE, dtype=np.uint8)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        obs = self.env.reset()
        if isinstance(obs, tuple):
            obs = obs[0]
        return self._process_obs(obs), {}

    def step(self, action):
        action = np.asarray(action, dtype=np.float32).reshape(-1)
        obs, reward, done, info = self.env.step(action)
        return self._process_obs(obs), float(reward), bool(done), False, info

    def _process_obs(self, obs):
        obs = obs[obs.shape[0] // 2:, :, :]
        gray = cv2.cvtColor(obs, cv2.COLOR_RGB2GRAY)
        resized = cv2.resize(gray, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        return np.expand_dims(resized, axis=0).astype(np.uint8)

    def render(self):
        return self.env.render(mode='rgb_array')

    def close(self):
        self.env.close()


class CustomCNN(BaseFeaturesExtractor):
    def __init__(self, observation_space, features_dim=256):
        super().__init__(observation_space, features_dim)
        n_input_channels = observation_space.shape[0]
        self.cnn = nn.Sequential(
            nn.Conv2d(n_input_channels, 32, kernel_size=8, stride=4), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1), nn.ReLU(),
            nn.Flatten())
        with torch.no_grad():
            sample = torch.as_tensor(observation_space.sample()[None]).float()
            n_flatten = self.cnn(sample).shape[1]
        self.linear = nn.Sequential(nn.Linear(n_flatten, features_dim), nn.ReLU())

    def forward(self, observations):
        observations = observations.float() / 255.0
        return self.linear(self.cnn(observations))


print('Clases definidas: DuckieWrapper, CustomCNN')

def make_test_env():
    return DuckieWrapper(MAP_NAME)

test_env = DummyVecEnv([make_test_env])
test_env = VecFrameStack(test_env, n_stack=N_STACK)
model = PPO.load(MODEL_PATH)

print('Evaluando agente...')
obs = test_env.reset()
frames = []
total_reward = 0.0
for i in range(1000):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, info = test_env.step(action)
    total_reward += float(reward[0])
    frames.append(test_env.envs[0].env.render(mode='rgb_array'))
    if done[0]:
        print(f'Episodio finalizado en el paso {i}')
        break

test_env.close()
print(f'Recompensa acumulada: {total_reward:.2f} | frames: {len(frames)}')
imageio.mimsave(VIDEO_PATH, frames, fps=30)
print(f'Video guardado: {VIDEO_PATH}')
"""

script = script.replace("MAP_NAME   = 'Duckietown-small_loop-v0'",
                        f"MAP_NAME   = '{MAP_NAME}'")

pathlib.Path('/tmp/eval_inline.py').write_text(script)

os.system('pkill -9 -f Xvfb 2>/dev/null; true')

# Un unico display X con GLX. La GL la resuelve Mesa software (LIBGL en el script).
cmd = f'xvfb-run -a -s "-screen 0 1024x768x24 +extension GLX +render -noreset" {PY} -u /tmp/eval_inline.py'
print('Evaluando agente...')
proc = subprocess.Popen(
    cmd, shell=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

if proc.returncode == 0 and os.path.exists(VIDEO_PATH):
    print('\nVideo guardado: ' + VIDEO_PATH)
    display(Video(VIDEO_PATH, embed=True))
else:
    print('\nERROR (codigo ' + str(proc.returncode) + '). Revisa el output arriba.')